# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load both the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata
# Display dataset title and description
print(metadata_obj.name + ": " + metadata_obj.description)

## 2. Data Overview
Review available record sets and their fields, referencing entities by their `@id` fields.
We will inspect the available record sets, fields, and columns as defined in the dataset schema.

In [ ]:
# Get available record sets and their @ids
record_sets = dataset.record_sets
record_set_ids = [rs['@id'] for rs in record_sets]

print("Available Record Sets:")
for rs in record_sets:
    print(f"  @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# Show fields for each record set
print("\nFields in each Record Set:")
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    elif not fields:
        print("  No fields defined.")
        continue
    for f in fields:
        print(f"    Field @id: {f['@id']} | name: {f.get('name', 'N/A')} | dataType: {f.get('dataType', 'N/A')}")
    print()


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data for each record set
# Reference entities by @id
dataframes = dict()

for rs in record_sets:
    record_set_id = rs['@id']
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) == 0:
            print(f"No records found for record set {record_set_id}")
        else:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"DataFrame loaded for {record_set_id}. Columns:")
            print(df.columns.tolist())
            print(df.head(), "\n")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing, and grouping records by key attributes. Reference fields and columns by their exact `@id`.

In [ ]:
# Choose a record set for analysis
if len(dataframes) > 0:
    analysis_record_set_id = list(dataframes.keys())[0]
    df = dataframes[analysis_record_set_id]
    print(f"Analyzing record set @id: {analysis_record_set_id}\nColumns: {df.columns.tolist()}\n")

    # Identify a likely numeric field based on column name
    numeric_field_candidates = [col for col in df.columns if 'Age' in col or 'age' in col or 'Interval' in col or 'interval' in col]
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
        print(f"Selected numeric field: {numeric_field_id}")
        
        threshold = 10
        if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records where {numeric_field_id} > {threshold}:")
            print(filtered_df.head())

            # Normalize the numeric field
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        else:
            print(f"Field {numeric_field_id} is not numeric!")
    else:
        print("No numeric field found for analysis. Please check column names.")

    # Group by another field (e.g., MSI status) if exists
    group_candidates = [col for col in df.columns if 'MSI' in col or 'msi' in col or 'Sex' in col or 'sex' in col or 'Subtype' in col]
    if group_candidates:
        group_field_id = group_candidates[0]
        print(f"Grouping by {group_field_id}:")
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(grouped_df.head())
        else:
            print(f"Field {group_field_id} not found in filtered DataFrame.")
    else:
        print("No categorical group field found for analysis.")
else:
    print("No dataframes available for EDA. Please review earlier steps.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and 'filtered_df' in locals():
    # Histogram of normalized numeric field
    if numeric_field_id in filtered_df.columns and f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=15, kde=True)
        plt.title(f"Normalized Distribution of {numeric_field_id}")
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.ylabel("Count")
        plt.show()

    # Boxplot between numeric field and group_field
    if group_field_id in filtered_df.columns and numeric_field_id in filtered_df.columns:
        plt.figure(figsize=(8, 6))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No DataFrame available for visualization.")

## 6. Conclusion
In this notebook, we've explored the FAIR² dataset and loaded its metadata and records, referencing all entities by their unique `@id`. We've performed basic EDA, filtered and normalized values, grouped by clinical attributes, and visualized key distributions. This workflow provides a foundation for further clinical or biomarker stratification studies using Croissant-structured datasets.